In [2]:
import sys
import os
from collections import defaultdict

# Add the src directory to sys.path
sys.path.append(os.path.abspath("../src"))

import torch
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown

from abstractions.dsl.abstraction import Abstraction
from abstractions.dsl.core import Shape
from abstractions.data.generator import generate_dataset
from abstractions.learning.utils import add, get_singletons, get_pairs
from abstractions.primitives.visualization import show_boxes, print_tree
from abstractions.learning.abstraction_discovery import find_abstractions, integrate_abstractions
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import hdbscan # Make sure you have this installed: pip install hdbscan
from sklearn.neighbors import NearestNeighbors

In [3]:
dataset = (
    generate_dataset("chair_1", 512) +
    generate_dataset("chair_2", 512) +
    generate_dataset("chair_3", 512)
)

print(f"Total shapes generated: {len(dataset)}")

Total shapes generated: 1536


In [4]:
slider = widgets.IntSlider(min=0, max=len(dataset)-1, step=1, description="Shape Index")
output_shape = widgets.Output()

def show_shape(idx):
    output_shape.clear_output()
    with output_shape:
        print(f"Displaying shape index: {idx}")
        print_tree(dataset[idx])

widgets.interact(show_shape, idx=slider)
display(output_shape)

interactive(children=(IntSlider(value=0, description='Shape Index', max=1535), Output()), _dom_classes=('widge…

Output()

In [5]:
# Organize structures
structures = {
    "Rect": [s for s in dataset if isinstance(s, Rect)],
    "Move": [s for s in dataset if isinstance(s, Move)]
}

def make_df_from_structure(param_list):
    float_data = extract_floats(param_list)
    float_data = [row for row in float_data if len(row) > 0]
    if not float_data:
        return pd.DataFrame()
    df = pd.DataFrame(float_data, columns=[f"param_{i}" for i in range(len(float_data[0]))])
    return df

dropdown = widgets.Dropdown(
    options=list(structures.keys()),
    description="Structure:",
    layout=widgets.Layout(width="50%")
)
output_params = widgets.Output()

def on_structure_change(change):
    structure_name = change['new']
    output_params.clear_output()
    with output_params:
        df = make_df_from_structure(structures[structure_name])
        display(Markdown(f"### `{structure_name}` – {len(df)} instances"))
        if df.empty:
            print("No float parameters available.")
        else:
            fig = px.scatter(df, x="param_0", y="param_1" if df.shape[1]>1 else None,
                             title=f"Scatterplot of `{structure_name}` parameters",
                             hover_data=df.columns)
            fig.show()

dropdown.observe(on_structure_change, names='value')
display(dropdown, output_params)
dropdown.value = list(structures.keys())[0]

NameError: name 'Rect' is not defined